### Generate OpenAlex Match DB

In [ ]:
import pandas as pd
import yaml
from tqdm import tqdm
from rapidfuzz import process, fuzz

In [ ]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False)

In [ ]:
import os
os.chdir('../../')

In [ ]:
def clean_string(text):
    import re
    if not text:
        return None
    text = text.lower()
    # Remove special characters except letters (a-z), digits, whitespace, and Chinese characters
    text = re.sub(r'[^a-z0-9\s\u4e00-\u9fff]', ' ', text)
    text = re.sub(r'[\d\s]+$', ' ', text)
    # Replace multiple spaces with a single space and trim leading/trailing spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def clean_author(text):
    import re
    if not text:
        return None

    # Remove text after "et al"
    match = re.search(r'\bet\s+al\b', text, flags=re.IGNORECASE)
    if match:
        cleaned = text[:match.start()]
    else:
        cleaned = text
    # Remove trailing ASCII/Chinese commas, the Chinese and-others marker, and extra whitespace
    cleaned = re.sub(r'[.,，等\s]+$', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    # Replace comma (and surrounding spaces) between two Chinese characters with a space
    cleaned = re.sub(r'([\u4e00-\u9fff])\s*,\s*([\u4e00-\u9fff])', r'\1 \2', cleaned)
    # Remove comma immediately following a Chinese character if not followed by another Chinese character
    cleaned = re.sub(r'([\u4e00-\u9fff])\s*,\s*(?![\u4e00-\u9fff])', r'\1', cleaned)
    # Remove comma immediately preceding a Chinese character if not preceded by another Chinese character
    cleaned = re.sub(r'(?<![\u4e00-\u9fff])\s*,\s*([\u4e00-\u9fff])', r'\1', cleaned)
    # Remove any extra whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    return cleaned

def contains_chinese(text):
    import re
    return bool(re.search(r'[\u4e00-\u9fff]', text))

In [ ]:
# Load configuration settings from a YAML file.
# The configuration file typically contains file paths and other parameters.
dataset_config = yaml.safe_load(open("config/dataset.yml"))
dataset_config

In [ ]:
openalex_title_year_full_path = os.environ.get(
    "OPENALEX_TITLE_YEAR_FULL_PARQUET",
    os.path.join(dataset_config["path_openalex"], "proc_datasets", "work_title_year_full.parquet"),
)

In [ ]:
CNROS_patent_paper = pd.read_parquet(dataset_config['path_processed'] + 'CN_CN/CN1_NPL_IncoPat_Formatted.parquet', engine='pyarrow')
CNROS_patent_paper

In [ ]:
CNROS_patent_paper['paper_title'] = CNROS_patent_paper['paper_title'].parallel_apply(clean_string)
CNROS_patent_paper['authors'] = CNROS_patent_paper['authors'].parallel_apply(clean_author)
CNROS_patent_paper['journal_name'] = CNROS_patent_paper['journal_name'].parallel_apply(clean_string)
CNROS_patent_paper = CNROS_patent_paper.dropna(subset=['paper_title'])
CNROS_patent_paper['is_chinese'] = CNROS_patent_paper['paper_title'].parallel_apply(contains_chinese)
CNROS_patent_paper

In [ ]:
CNROS_patent_paper[CNROS_patent_paper.year.isna()]

In [ ]:
oa_works = pd.read_parquet(openalex_title_year_full_path).rename(columns={'id': 'work_id', 'display_name': 'paper_title', 'publication_year': 'year'})
oa_works

In [ ]:
pandarallel.initialize(progress_bar=True)
oa_works['paper_title'] = oa_works['paper_title'].parallel_apply(clean_string)

In [ ]:
#oa_authors = pd.read_parquet('proc_datasets/work_author_names_full.parquet')
#oa_journals = pd.read_parquet('proc_datasets/journal_name_full.parquet').rename(columns={'primary_location/source/display_name': 'journal_name'})

In [ ]:
oa_works.paper_title

In [ ]:
process.extractOne("Comparative Genomic Analysis and BTEX", oa_works.paper_title, scorer=fuzz.QRatio)